<a href="https://colab.research.google.com/github/aiwithpavansama/GenerativeandAgenticAI/blob/main/resume_matching_using_SentenceTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.listdir('/content/drive/MyDrive')   # list files in your Drive root



In [ ]:
!pip install pypdf

from pypdf import PdfReader

reader = PdfReader('/content/drive/MyDrive/resumes/data_science_01_david_kim.pdf')

# all pages
text = ""
for page in reader.pages:
    text += page.extract_text() or ""

print(text)

In [ ]:
!pip install pypdf sentence-transformers scikit-learn

import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FOLDER = '/content/drive/MyDrive/resumes'

def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

# read ALL pdfs
files = sorted(f for f in os.listdir(FOLDER) if f.lower().endswith('.pdf'))
texts = [read_pdf(os.path.join(FOLDER, f)) for f in files]

# embed all
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, convert_to_numpy=True)

# match first resume against the rest
sims = cosine_similarity([embeddings[0]], embeddings)[0]
order = [i for i in np.argsort(sims)[::-1] if i != 0]   # drop the query itself

# show 5 best matches with name + similarity
for rank, i in enumerate(order[:5], 1):
    print(f"Best match {rank}: {files[i]}  (similarity: {sims[i]:.3f})")